# Урок 7. Pandas: таблица и исследование пассажиров

**Сегодня:** прочитаем знакомый Titanic в Pandas, проверим таблицу, выберем группы пассажиров и сопоставим их показатели.

Готовые демонстрационные ячейки запускайте по порядку. В практиках и ДЗ пишите код в подготовленных ячейках; `? ВЫВОДЫ` — место для вашего объяснения результата. Не нужно добавлять новые рабочие ячейки.

**Для запуска:** этот ноутбук и `titanic.csv` рядом с ним. В Colab загрузите CSV в панель «Файлы». Локально используйте настроенное окружение курса. Подготовка файла ниже — служебная ячейка; её код разбирать на занятии не требуется.

## Знакомимся с данными: пассажиры «Титаника»

«Титаник» вышел из Саутгемптона в Нью-Йорк 10 апреля 1912 года. После столкновения с айсбергом корабль затонул 15 апреля. [Историческая справка: Royal Museums Greenwich](https://www.rmg.co.uk/collections/research-guides/research-guide-d1-rms-titanic-fact-sheet).

Мы работаем с учебной таблицей из **891 записи о пассажирах**, а не с полным списком всех людей на борту. Одна строка — запись об одном пассажире. В ней есть характеристики пассажира и признак `Survived`: 1 — выжил, 0 — не выжил.

**Зачем этот набор в курсе?** На небольшом файле можно пройти весь путь анализа: прочитать таблицу, понять поля, найти пропуски, выбрать группы, сравнить показатели, сохранить результат и позднее построить модель. Сегодня исследуем, как различаются доли выживших в выбранных группах. Наблюдаемое различие не объясняет само по себе причины катастрофы или исход для каждого человека.

**Почему встречаются пустые поля?** Пустое поле означает: значение не указано в этой учебной таблице. По CSV нельзя установить причину каждого пропуска. Значение могли не записать или не включить в набор — это возможные объяснения, а не установленная история конкретного пассажира. Неизвестный возраст не равен нулю; неизвестная каюта не означает, что пассажир ехал без каюты. Мы сохраняем эту неопределённость и учитываем её при расчётах.

Источник таблицы — [учебный набор pandas](https://pandas.pydata.org/docs/getting_started/intro_tutorials/02_read_write.html). Его описание полей и примеры помогут сверить смысл данных.


In [12]:
#@title Подготовка данных — запустите один раз
from pathlib import Path
import hashlib

# Сначала используем выданный файл. Интернет нужен только если файла нет.
DATA_NAME = "titanic.csv"
DATA_SHA256 = "4a437fde05fe5264e1701a7387ac6fb75393772ba38bb2c9c566405af5af4bd7"
candidates = [Path(DATA_NAME), Path("data") / DATA_NAME]
for base in [Path.cwd(), *Path.cwd().parents][:5]:
    candidates.extend([
        base / "07_Данные" / "lesson-07" / DATA_NAME,
        base / "01_Course" / "Python для ИИ" / "07_Данные" / "lesson-07" / DATA_NAME,
    ])
DATA_PATH = next((p for p in candidates if p.is_file()), None)
if DATA_PATH is None:
    from urllib.request import urlopen
    try:
        raw = urlopen("https://raw.githubusercontent.com/pandas-dev/pandas/v2.3.3/doc/data/titanic.csv", timeout=30).read()
    except OSError as error:
        raise FileNotFoundError("Положите выданный titanic.csv рядом с ноутбуком. В Colab загрузите его в панель «Файлы».") from error
    if hashlib.sha256(raw).hexdigest() != DATA_SHA256:
        raise ValueError("Источник данных изменился. Используйте файл, выданный преподавателем.")
    DATA_PATH = Path(DATA_NAME)
    # Режим xb не позволяет случайно перезаписать существующий файл.
    with DATA_PATH.open("xb") as file:
        file.write(raw)
if hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() != DATA_SHA256:
    raise ValueError("Это другая версия titanic.csv. Сохраните её отдельно и используйте выданный учебный файл.")
print("Данные готовы:", DATA_PATH.name)


Данные готовы: titanic.csv


**Подготовка завершена:** дальше читаем содержимое файла. Проверка версии CSV нужна, чтобы у группы были одинаковые исходные данные; она не заменяет проверку записей.

## 🟨❗ Шаг 1. Прочитать тот же файл как таблицу

В уроке 6 CSV читался в словари со строковыми значениями; затем мы явно создавали массивы. Теперь `pd.read_csv()` читает файл в `DataFrame` и **сам определяет типы столбцов** по содержимому. Пустые значения он обозначает как пропуски. Это удобно, но результат надо проверить.

Сначала загрузим df и посмотрим пять строк. После чтения пройдём отдельный этап знакомства с таблицей: начало и конец, размеры, поля, типы, пропуски и сводки. `head(5)` — пять примеров, а не вся выборка.

In [13]:
import pandas as pd
df = pd.read_csv(DATA_PATH)
display(df.head(5))

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 🟨❗ Шаг 2. Знакомство с таблицей перед исследованием — 13 минут

`shape` возвращает число строк и столбцов. `columns` — их названия. `dtypes` — тип каждого столбца: целые числа, дробные числа или текст. В разных версиях pandas текст может обозначаться как `object` или `str`; важен смысл поля, а не точное написание типа.

Survived хранится как число, но означает 0/1. Pclass — категория класса билета. Age — возраст в годах с пропусками, Fare — записанная плата за проезд. Дробный тип Age позволяет хранить дробные возраста и NaN.

Сначала назовём размер и поля; затем последовательно выполним осмотр ниже. После каждой команды объясняем результат и решаем, что он позволяет проверить.


In [14]:
print("Строк и столбцов:", df.shape)
print("Названия полей:", df.columns.tolist())
print("Типы столбцов:")
print(df.dtypes)

Строк и столбцов: (891, 12)
Названия полей: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']
Типы столбцов:
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object


### Что означают поля, с которыми работаем сегодня

| Поле в CSV | По-русски | Пример и смысл |
|---|---|---|
| PassengerId | Идентификатор пассажира | Число связывает запись с исходной таблицей; это не индекс массива |
| Survived | Признак выживания | 0 или 1; сумму можно использовать для подсчёта выживших |
| Pclass | Класс билета | 1, 2 или 3; число обозначает категорию |
| Name, Sex | Имя и пол из таблицы | Текстовые значения; Sex записан как female/male |
| Age | Возраст в годах | Может быть дробным или отсутствовать |
| Fare | Плата за проезд, указанная в записи | Числовой показатель стоимости, а не номер билета |

Номер билета хранится в отдельном поле `Ticket`; каюта — в `Cabin`, порт посадки — в `Embarked`. Полный словарь 12 полей находится в `README_Titanic.md`.

**Как читать Fare:** это записанная в наборе плата за проезд. В самом CSV нет указания денежной единицы и правил распределения стоимости общего билета между пассажирами. Поэтому в заданиях называем показатель Fare, сравниваем его значения и не переводим их в современные цены. Нулевое значение само по себе не доказывает ошибку.


### Осмотр 1. Начало и конец таблицы

`head(5)` показывает первые пять строк, `tail(5)` — последние пять. Число в скобках можно менять. Без аргумента оба метода показывают пять строк. Они не изменяют `df` и не показывают весь набор данных.

**Зачем:** увидеть, как устроены записи в разных частях файла. Первые строки не являются случайной выборкой и могут не содержать ни одного пропуска, хотя в остальной таблице пропуски есть.

**Вопрос:** последние пять строк имеют те же поля, что первые? Можно ли по десяти показанным строкам утверждать, что во всём наборе нет неизвестного возраста?


In [15]:
print("Последние пять записей — пример, не вся таблица:")
display(df.tail(5))


Последние пять записей — пример, не вся таблица:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.00,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.00,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.45,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.00,C148,C
890,891,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,370376,7.75,NaN,Q


### Осмотр 2. Краткая информация о всей таблице

`df.info()` выводит названия столбцов, типы хранения, число непустых значений (`Non-Null Count`) и сведения о памяти. Здесь в каждой строке таблицы должен быть пассажир; количество непустых значений отдельного поля может быть меньше количества пассажиров.

**Как читать:** сравните `Non-Null Count` с числом строк из `df.shape`. Если значений Age меньше, часть возрастов неизвестна. Раздел `Dtype` показывает, как Pandas хранит поле, но не определяет его смысл: числовой `Pclass` остаётся категорией, а `PassengerId` — идентификатором.

`info()` само печатает сводку и возвращает `None`; вызываем `df.info()`, без `print(df.info())`. Поля таблицы — `columns`, типы — `dtypes`, размер — `shape`: у этих атрибутов скобок нет. У методов `head()`, `tail()` и `info()` скобки нужны.

**Вопрос:** почему количество непустых значений Age и Cabin отличается от числа строк? Достаточно ли этого, чтобы удалить такие записи?


In [16]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


### Осмотр 3. Посчитать пропуски по каждому полю

`df.isna()` отмечает пропущенные значения; `.sum()` считает эти отметки отдельно в каждом столбце. Эта проверка охватывает всю таблицу, а не только строки из `head()`.

**Зачем:** определить, какие расчёты требуют отбора известных значений. Неизвестный Age мешает возрастному сравнению, но не требует исключать пассажира из сравнения классов, если Pclass и Survived известны. Неизвестный Cabin не означает, что пассажир ехал без каюты.

**Вопрос:** в каких полях есть пропуски? Сколько записей можно использовать для среднего возраста? Не заполняйте пропуски и не удаляйте строки на этом шаге.


In [17]:
print("Пропущенных значений по столбцам:")
display(df.isna().sum())


Пропущенных значений по столбцам:


,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


### Осмотр 4. Числовая сводка: возраст и плата за проезд

`describe()` даёт сводку числовых полей. Выберем `Age` и `Fare`, потому что их числовые характеристики содержательно полезны. Среднее идентификатора PassengerId и среднее номера класса Pclass не отвечают нашим исследовательским вопросам.

Сегодня читаем строки `count` (сколько значений известно), `mean` (среднее), `min` и `max`. Пропуски не входят в эти расчёты. Полный вывод также содержит `std` — стандартное отклонение и `25%`, `50%`, `75%` — квартили; подробно разберём их на занятии об описательной статистике. `50%` — медиана, она не обязана совпадать со средним.

**Зачем:** заметить диапазон значений и понять, по скольким наблюдениям рассчитана сводка. Нулевой Fare или большой максимум — повод посмотреть запись и источник, а не доказательство ошибки.

**Вопрос:** почему `count` у Age и Fare различается? Какую запись следует изучить отдельно, прежде чем решать, что значение Fare ошибочно?


In [18]:
display(df[["Age", "Fare"]].describe())


,Age,Fare
count,714.000000,891.000000
mean,29.699118,32.204208
std,14.526497,49.693429
min,0.420000,0.000000
25%,20.125000,7.910400
50%,28.000000,14.454200
75%,38.000000,31.000000
max,80.000000,512.329200


### Осмотр 5. Категории и число записей в каждой

`value_counts()` для одного столбца показывает встречающиеся значения и их частоты. Применим его к классу билета и порту посадки. По умолчанию пропуски не входят в сводку; `dropna=False` позволяет увидеть их отдельной строкой. Результат обычно упорядочен по частоте, а не по порядку пассажиров.

**Зачем:** проверить, какие категории реально есть в выбранном поле и сколько записей относится к каждой. Это сводка состава данных, ещё не сравнение выживаемости.

**Вопрос:** почему строка с неизвестным Embarked полезна для проверки? Можно ли по числу пассажиров каждого класса определить долю выживших в классе?


In [19]:
print("Класс билета: число пассажиров")
display(df["Pclass"].value_counts(dropna=False))
print("Порт посадки: число пассажиров, включая неизвестный порт")
display(df["Embarked"].value_counts(dropna=False))


Класс билета: число пассажиров


,count
Pclass,
3,491
1,216
2,184


Порт посадки: число пассажиров, включая неизвестный порт


,count
Embarked,
S,644
C,168
Q,77
NaN,2


### Итог осмотра: что узнали и что ещё не делаем

Перед исследованием называем единицу наблюдения, размер таблицы, нужные поля, типы и пропуски. `head()` и `tail()` показывают примеры; `info()` и подсчёт пропусков описывают весь набор. Ни одна из этих команд сама по себе не доказывает, что данные полностью корректны.

Сейчас диагностируем данные и сохраняем исходную `df`. Преобразование типов, обработку пропусков и удаление подтверждённых дублей будем обосновывать на соответствующих следующих уроках. Для текущего вопроса можно создать отдельную выборку, не изменяя исходник.


## 🟨❗ Один столбец и таблица: Series и DataFrame

`DataFrame` — таблица с названиями столбцов и индексом строк. `Series` — один столбец с индексом. `df["Survived"]` возвращает Series; `df[["Survived"]]` возвращает таблицу из одного столбца. Внутренние квадратные скобки во втором варианте создают список названий, внешние выбирают эти столбцы из df.

Посмотрим оба результата на первых трёх записях. Данные те же, форма представления разная.

In [20]:
print("Один столбец как Series:")
display(df["Survived"].head(3))
print("Один столбец как DataFrame:")
display(df[["Survived"]].head(3))

Один столбец как Series:


,Survived
0,0
1,1
2,1


Один столбец как DataFrame:


,Survived
0,0
1,1
2,1


## ⭐ Практика 1. Паспорт таблицы

Выведите число строк и столбцов, названия всех столбцов. Выберите Series Survived, затем DataFrame из PassengerId, Pclass, Age и Survived и покажите первые пять строк. Выведите тип каждого результата через type(...).


**Зачем этот шаг:** Размер и названия обнаруживают неверный файл, разделитель или потерю столбцов. Тип результата помогает выбрать дальнейшую операцию: один столбец и таблица ведут себя по-разному. Первые строки — только осмотр.

Дополните паспорт: вызовите `info()` и выведите число пропусков Age. Назовите число известных возрастов. Своими словами объясните, почему head/tail недостаточно для проверки всей таблицы и почему на этом шаге мы не удаляем строки.


In [21]:
import pandas as pd
df = pd.read_csv('titanic.csv')
print(df.shape)
print(df.columns.tolist())
survived = df['Survived']
print(survived.head())
print(type(survived))
subset = df[['PassengerId', 'Pclass', 'Age', 'Survived']]
print(subset.head())
print(type(subset))



(891, 12)
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']
0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64
<class 'pandas.core.series.Series'>
   PassengerId  Pclass   Age  Survived
0            1       3  22.0         0
1            2       1  38.0         1
2            3       3  26.0         1
3            4       1  35.0         1
4            5       3  35.0         0
<class 'pandas.core.frame.DataFrame'>


In [22]:
print("Строк и столбцов:", df.shape)
print("Названия полей:", df.columns.tolist())
print("Типы столбцов:")
print(df.dtypes)

Строк и столбцов: (891, 12)
Названия полей: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']
Типы столбцов:
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object


In [23]:
print("Строк и столбцов:", df.shape)
print("Названия полей:", df.columns.tolist())
print("Типы столбцов:")
print(df.dtypes)

Строк и столбцов: (891, 12)
Названия полей: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']
Типы столбцов:
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object


### ❓ Вопросы и выводы

Какой тип вернул `df["Survived"]`, а какой — `df[["PassengerId", "Pclass", "Age", "Survived"]]`? Объясните: внутри вторых скобок находится список названий столбцов.

Что показало info об Age? Сколько строк останется в исходной таблице после нашего осмотра? Приведите один пример поля, которому пропуск Age не мешает.


? ВЫВОДЫ

Ответьте на вопрос выше. Если вы рассчитали показатель, назовите группу, число записей и результат; затем объясните, что из него следует.

## 🟨❗ Шаг 3. Выбрать пассажиров и нужные поля

`df.loc[условие, список_столбцов]` выбирает строки по условию и показывает нужные поля. Сначала посмотрим пассажиров первого класса. Затем соединим два условия: `(df["Pclass"] == 3) & (df["Age"] >= 18)`. Скобки сохраняют границы каждого сравнения; для Series используем `&`, а не `and`.

`.iloc` выбирает по позиции: `df.iloc[:3, :2]` — первые три строки и два столбца. Позиция 0 и PassengerId 1 — разные понятия.

In [24]:
display(df.loc[df["Pclass"] == 1, ["PassengerId", "Name", "Pclass"]].head(3))
print("Первые три строки и два столбца по позиции:")
display(df.iloc[:3, :2])

,PassengerId,Name,Pclass
1,2,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1
3,4,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1
6,7,"McCarthy, Mr. Timothy J",1


Первые три строки и два столбца по позиции:


,PassengerId,Survived
0,1,0
1,2,1
2,3,1


## ⭐ Практика 2. Отбор и граница

Выберите взрослых (Age >= 18) пассажиров третьего класса. Покажите первые пять строк только PassengerId, Name, Age, Survived; выведите общее число отобранных строк. Затем во всей исходной таблице df посчитайте число пассажиров с Age == 18, независимо от класса билета.


**Зачем этот шаг:** Проверка возраста ровно на границе выявляет ошибку < / <=. Полный размер группы защищает от расчёта по пяти показанным строкам. Неизвестный возраст исключает запись из возрастного сравнения, но не из всего исследования.

In [25]:
import pandas as pd
df = pd.read_csv('titanic.csv')
adults_3rd = df[(df['Age'] >= 18) & (df['Pclass'] == 3)]
print(adults_3rd[['PassengerId', 'Name', 'Age', 'Survived']].head())
print('Взрослых в 3-м классе:', len(adults_3rd))
print('Пассажиров с Age == 18 (вся таблица):', (df['Age'] == 18).sum())

    PassengerId                                               Name   Age  \
0             1                            Braund, Mr. Owen Harris  22.0   
2             3                             Heikkinen, Miss. Laina  26.0   
4             5                           Allen, Mr. William Henry  35.0   
8             9  Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)  27.0   
12           13                     Saundercock, Mr. William Henry  20.0   

    Survived  
0          0  
2          1  
4          0  
8          1  
12         0  
Взрослых в 3-м классе: 277
Пассажиров с Age == 18 (вся таблица): 26


### ❓ Вопросы и выводы

Включает ли условие `Age >= 18` пассажиров ровно 18 лет? Во всей таблице или только в третьем классе вы посчитали `Age == 18`? Почему пять показанных строк не равны числу людей в выбранной группе?

? ВЫВОДЫ

Ответьте на вопрос выше. Если вы рассчитали показатель, назовите группу, число записей и результат; затем объясните, что из него следует.

## 🟨❗ Шаг 4. Посчитать строки, известные значения и среднее

`len(df)` считает все строки. `df["Age"].count()` — только значения Age без пропусков; `.isna().sum()` — число пропусков. `.notna()` создаёт маску записанных значений.

В Pandas `.mean()` пропускает отсутствующие значения по умолчанию. Для Age среднее поэтому относится к записям с указанным возрастом. В отличие от обычного NumPy mean, результат может быть числом даже при наличии пропусков — знаменатель всё равно нужно назвать.

Для Survived из значений 0/1 без пропусков `.sum()` — число выживших, `.mean()` — их доля. Вспомните пример 2 выживших из 4: 2/4 = 0.5 = 50%.

In [26]:
example = pd.Series([12.0, None, 30.0])
print("Строк / известных:", len(example), example.count())
print("Среднее известных:", example.mean())


Строк / известных: 3 2
Среднее известных: 21.0


## ⭐ Практика 3. Повторить расчёт NumPy

Выведите число пассажиров, число выживших и их долю. Для Age выведите число известных значений, число пропусков и средний возраст. Сверьте с уроком 6: 891 запись, 342 выживших, 714 известных возрастов. Это контроль загрузки, а не готовый вывод.


**Зачем этот шаг:** Сверка с прошлым уроком проверяет, что инструменты работают с теми же данными. Разница между числом строк и числом известных возрастов объясняет знаменатель среднего; пропуски нельзя молча считать нулевыми возрастами.

In [27]:
import pandas as pd
df = pd.read_csv('titanic.csv')
n_passengers = len(df)
n_survived   = int(df['Survived'].sum())
survived_share = df['Survived'].mean()
print('Всего пассажиров:', n_passengers)
print('Выживших:', n_survived)
print('Доля выживших:', round(survived_share, 4))
n_known_age = df['Age'].notna().sum()
n_missing_age = df['Age'].isna().sum()
mean_age = df['Age'].mean()

print('Известных возрастов:', n_known_age)
print('Пропусков Age:', n_missing_age)
print('Средний возраст:', round(mean_age, 2))

Всего пассажиров: 891
Выживших: 342
Доля выживших: 0.3838
Известных возрастов: 714
Пропусков Age: 177
Средний возраст: 29.7


### ❓ Вопросы и выводы

Чем len(df) отличается от df["Age"].count()? Согласуются ли числа с прошлым уроком?


? ВЫВОДЫ

Ответьте на вопрос выше. Если вы рассчитали показатель, назовите группу, число записей и результат; затем объясните, что из него следует.

## ⭐ Практика 4. Исследование: первый и третий класс

До запуска сформулируйте гипотезу о доле выживших в классах 1 и 3. Для каждого класса через фильтр выведите n, выживших, долю и среднее значение Fare (плата за проезд). Можно использовать знакомый цикл по [1, 3]; groupby пока не требуется.


**Зачем этот шаг:** Одинаковые показатели позволяют сравнить классы. Размер каждой группы проверяет знаменатель доли и показывает объём наблюдений; различие долей само по себе не доказывает причину.

In [28]:
# Моя гипотеза: ...
# Два класса, одинаковые показатели и разные маски.


### ❓ Вопросы и выводы

Сравните доли выживших и размеры групп. Подтвердилось ли ваше предположение для этой таблицы? Почему более высокая доля в группе не означает, что каждый её пассажир выжил?

? ВЫВОДЫ

Ответьте на вопрос выше. Если вы рассчитали показатель, назовите группу, число записей и результат; затем объясните, что из него следует.

## 🟨❗ Шаг 5. Рассмотреть записи после отбора

`sort_values("Fare", ascending=False)` упорядочивает записи по значению платы за проезд Fare: от большего к меньшему. `.head(5)` показывает первые пять строк этого результата. Число всех отобранных пассажиров считаем отдельно.

Сортировка без присваивания не меняет исходный df. Она помогает найти примеры для обсуждения, но не объясняет причину исхода.

## ⭐ Практика 5. Контрпример и самостоятельный вывод

Отберите пассажиров первого класса, которые не выжили. Отсортируйте их по Fare по убыванию, покажите пять строк с PassengerId, Name, Fare, Survived и выведите полное число в группе. Затем сформулируйте один свой вопрос о Pclass/Age/Survived и выполните короткий расчёт в той же Code-ячейке.


**Зачем этот шаг:** Контрпример проверяет утверждение о гарантированном исходе. Сортировка помогает рассмотреть конкретные записи, но пять записей с наибольшим Fare не описывают всех пассажиров. Свой вопрос проверяет, можете ли вы самостоятельно выбрать группу и показатель.

In [29]:
# Отбор, сортировка и свой вопрос.


### ❓ Вопросы и выводы

Запишите свой вопрос, численный ответ и ограничение. Что контрпример меняет в формулировке вывода о классе билета?


? ВЫВОДЫ

Ответьте на вопрос выше. Если вы рассчитали показатель, назовите группу, число записей и результат; затем объясните, что из него следует.

# 🏠 Домашнее задание. Короткое исследование Titanic

Обязательны H1–H3, H4 дополнительный. Используйте исходный df, не удаляйте столбцы
и не заполняйте пропуски. В каждом шаге нужны расчёт и вывод. Исследование: 40–55 минут; сохранение и описание файлов: ещё 5–10 минут.


## 🏠 Домашнее задание H1. Известный возраст и две группы

Выделите `known` только с указанным Age через notna(). Внутри этой выборки сравните группы с Age < 18 и Age >= 18 по n, числу выживших и доле. Выведите число исключённых строк относительно исходного df.


**Зачем этот шаг:** Фильтр известного возраста делает ограничение выборки явным. Подсчёт исключённых строк позволяет объяснить, кого вывод не охватывает. Исходную таблицу сохраняем для вопросов, которым возраст не нужен.

In [30]:
# Фильтр notna и две возрастные группы.


### ❓ Вопросы и выводы

Сравните с NumPy: совпали ли n и доли? Объясните, почему знаменатель не 891.


? ВЫВОДЫ

Ответьте на вопрос выше. Если вы рассчитали показатель, назовите группу, число записей и результат; затем объясните, что из него следует.

## 🏠 Домашнее задание H2. Проверить общую закономерность внутри классов

Для каждого класса 1, 2, 3 среди записей с указанным возрастом сравните пассажиров младше 18 лет и от 18 лет. Выведите n, число выживших и долю для каждой из шести групп. Пустую группу, если возникнет, обозначьте «недостаточно данных». Используйте знакомый цикл и .loc; groupby не нужен.


**Зачем этот шаг:** Сравнение внутри классов проверяет, сохраняется ли общая закономерность в разных группах. Размеры защищают от переоценки процента по малой группе; в пустой группе долю определить нельзя.

In [31]:
# Класс + известный возраст + возрастная граница.


### ❓ Вопросы и выводы

Сохраняется ли направление различия во всех трёх классах? Выберите одну маленькую группу и объясните, почему её процент нельзя считать точным прогнозом.


? ВЫВОДЫ

Ответьте на вопрос выше. Если вы рассчитали показатель, назовите группу, число записей и результат; затем объясните, что из него следует.

## 🏠 Домашнее задание H3. Собственный исследовательский вопрос

Выберите вопрос, который можно решить изученными инструментами: например, меняется ли доля выживших внутри одного класса для порогов возраста 16 и 18; или как различаются средние значения Fare (плата за проезд) между выжившими пассажирами одного выбранного класса и пассажирами того же класса, которые не выжили. Перед кодом запишите вопрос и гипотезу комментарием. Выведите n и показатель для каждой сравниваемой группы.


**Зачем этот шаг:** Вопрос определяет нужные поля, фильтры и проверки. Для среднего Fare надо проверить его известные значения, для возраста — известный Age. Размер группы не всегда равен числу значений, использованных в среднем.

In [32]:
# Мой вопрос: ...
# Моя гипотеза: ...
# Расчёт с подписями и n.


### ❓ Вопросы и выводы

Запишите два вывода по числам и одно ограничение. Не подменяйте вопрос о связи утверждением о причине.


? ВЫВОДЫ

Ответьте на вопрос выше. Если вы рассчитали показатель, назовите группу, число записей и результат; затем объясните, что из него следует.

## 🏠 Домашнее задание H4. Дополнительно: сверка двух инструментов

Считайте долю выживших третьего класса в Pandas, затем преобразуйте только Series Survived выбранной группы в NumPy через .to_numpy() и посчитайте mean(). Напечатайте обе доли и их разность. Не объединяйте строковые и числовые столбцы в один массив.


**Зачем этот шаг:** Сверка помогает обнаружить разные маски, выборки или обработку пропусков. Совпадение чисел подтверждает согласованность расчётов, но не доказывает, что исходные данные достоверны.

In [33]:
# Одинаковая группа для двух инструментов.


### ❓ Вопросы и выводы

Обе доли рассчитаны для одних и тех же пассажиров третьего класса. Совпали ли они? Почему сравнение потеряет смысл, если в Pandas и NumPy выбрать разные группы?

? ВЫВОДЫ

Ответьте на вопрос выше. Если вы рассчитали показатель, назовите группу, число записей и результат; затем объясните, что из него следует.

## 🏠 Передача данных на следующий урок

После H1–H3 запустите готовую ячейку ниже. Она сохраняет всю таблицу и отчёт о пропусках в `data/`, затем проверяет повторное чтение. Это проверенная исходная версия, ещё без очистки. Если проверка сообщает, что df отличается от исходника, восстановите df из CSV и повторите исследования с отдельными выборками.

Создайте свой `data/README_lesson07.md`: источник; размеры; пропуски; что изменяли или почему сохранили без изменений; что будете проверять при очистке. Готовый экспорт не является ответом на исследовательские задания. Скачайте три результата из Colab и загрузите вместе с README в `homework/lesson-07/data/`. На уроке 8 начнёте с собственного `titanic_lesson07.csv`.

Ориентир переноса и описания: ещё 5–10 минут. Полный маршрут и паспорт полей — в `README_Titanic.md` из студенческого комплекта.

In [34]:
#@title Сохранить данные для следующего урока
# Готовая служебная ячейка: создаёт файлы для следующего урока.
# Она не заменяет расчёты и выводы в H1–H3.
import json
import hashlib
from pathlib import Path

# В уроке 7 исходную таблицу ещё не изменяем.
expected_raw = pd.read_csv(DATA_PATH)
try:
    pd.testing.assert_frame_equal(df, expected_raw)
except AssertionError as error:
    raise ValueError("Исходная df изменена. Заново прочитайте CSV и выполняйте исследования в отдельных выборках.") from error
output_dir = Path("data")
output_dir.mkdir(exist_ok=True)
result_path = output_dir / "titanic_lesson07.csv"
df.to_csv(result_path, index=False, encoding="utf-8")
reloaded = pd.read_csv(result_path)
pd.testing.assert_frame_equal(df, reloaded)
quality = pd.DataFrame({"field": df.columns,
                        "known": df.count().to_numpy(),
                        "missing": df.isna().sum().to_numpy()})
quality.to_csv(output_dir / "quality_lesson07.csv", index=False, encoding="utf-8")
manifest = {"stage": "lesson-07-checked-not-cleaned", "source": "titanic.csv",
            "source_sha256": hashlib.sha256(Path(DATA_PATH).read_bytes()).hexdigest(),
            "output": result_path.name,
            "output_sha256": hashlib.sha256(result_path.read_bytes()).hexdigest(),
            "rows": len(df), "columns": list(df.columns),
            "changes": "Без очистки; сохранение CSV без индекса; пропуски сохранены"}
(output_dir / "manifest_lesson07.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("Сохранены и проверены: titanic_lesson07.csv, quality_lesson07.csv, manifest_lesson07.json")
print("Добавьте свой README_lesson07.md и сдайте папку data вместе с ноутбуком.")


Сохранены и проверены: titanic_lesson07.csv, quality_lesson07.csv, manifest_lesson07.json
Добавьте свой README_lesson07.md и сдайте папку data вместе с ноутбуком.


## 🏠 Домашнее задание. Проверка перед сдачей

1. Выполните все обязательные домашние шаги и заполните их `? ВЫВОДЫ`.
2. Нажмите Restart Kernel / Restart Runtime, затем Run All. Исправьте ошибки,
   убедитесь, что результаты соответствуют текущему коду, сохраните ноутбук с выводами.
3. В готовом решении не оставляйте обязательные Code-ячейки только с комментариями.
4. Загрузите также папку `data/` с CSV, отчётом, manifest и собственным README.
5. Загрузите файл по пути `homework/lesson-07/solution.ipynb` и проверьте его отображение в GitHub.
6. Срок преподаватель объявит отдельно. Новая дата в этом шаблоне не назначена.

**Зачем перезапускать и сохранять:** чистый запуск выявляет зависимость от старых переменных и случайного порядка выполнения; сохранённые выводы позволяют преподавателю сверить ответ с текущим кодом. Просмотр файла в GitHub проверяет, что загружена нужная версия и результаты видны.

**Что проверяется:** работающий код, верные фильтры и знаменатели, сохранённые результаты,
выводы по расчётам, объяснение ограничения. Дополнительное задание не заменяет обязательную часть.

**Следующий урок:** исследование и преобразование данных; заполнение пропусков сейчас не выполняем.

## Источники
- [Набор Titanic в документации pandas](https://raw.githubusercontent.com/pandas-dev/pandas/v2.3.3/doc/data/titanic.csv)
- [NumPy: основы](https://numpy.org/doc/stable/user/absolute_beginners.html)
- [Pandas: учебные руководства](https://pandas.pydata.org/docs/getting_started/intro_tutorials/)


## Справка. Проверки данных перед анализом

Начинайте с вопроса и полей, которые нужны для ответа. Затем проверяйте данные. Очистка начинается с найденной и объяснённой проблемы; после неё надо повторить проверки и посчитать, сколько записей изменилось.

| Проверка | Зачем | Решение после проверки |
|---|---|---|
| `shape`, `columns`, `head()`, `tail()` | Заметить другой файл, неверный разделитель или сдвиг полей | Исправить загрузку; head показывает пример, не подтверждает качество всей таблицы |
| `info()`, `dtypes` и смысл значений | Не считать среднее строк и не принимать код категории за измерение | Сверить исходные значения; преобразовать тип осмысленно на уроке преобразований |
| `.isna().sum()`, `.count()`, `.notna()` для нужного поля | Понять, кому не хватает данных и сколько значений войдёт в показатель | Указать пропуски и знаменатель; выбирать известные значения для конкретного вопроса |
| Допустимые категории и диапазоны | Проверить 0/1 в Survived, 1/2/3 в Pclass, неотрицательные известные Age/Fare | При нарушении сверить источник; правило исправления обосновать |
| Повторные записи и PassengerId | Не учитывать одного пассажира несколько раз | Выяснить причину повторов; одинаковые фамилия, билет или Fare ещё не означают дубль пассажира |
| Минимум/максимум и строки с необычным Fare | Найти значения, которые требуют проверки | Проверить запись и контекст; не удалять автоматически дорогие или бесплатные билеты |
| Число строк после фильтра и граница возраста | Не сравнить случайно разные группы | Уточнить маску; для пустой группы не рассчитывать процент |

Размер, имена и типы столбцов, пропуски и размеры групп проверяем изученными командами в этом уроке. Проверки категорий, дублей и подробные решения по очистке — ориентиры для урока 9; их автоматизацию сейчас добавлять не требуется.

**Как принимать решение об очистке:** записать проблему → сверить источник и смысл поля → выбрать действие → работать с отдельной копией → сравнить до/после и объяснить влияние на расчёт. Дубликат удаляют после подтверждения; формат и тип исправляют по понятному правилу; пропуск сохраняют, исключают из конкретного расчёта или заполняют с обоснованием. Универсального действия для всех проблем нет.

**Age с пропуском нельзя заменить нулём:** это превратит неизвестного пассажира в младенца. Удаление всех строк с пропусками тоже может изменить состав выборки без причины. Для вопроса о классе Age не обязателен. В уроках 6–7 исходную таблицу не чистим; диагностируем и явно ограничиваем выборку. Заполнение пропусков и оценка последствий — урок 9.


`describe()` по Age/Fare помогает прочитать count/mean/min/max; `value_counts(dropna=False)` — частоты категорий с пропусками. Диагностика не заменяет проверку допустимых значений и не является автоматической очисткой.
